# Comparação BERTimbau vs. Gemini Flash-Lite — Classificação de Intenção

Este notebook implementa o experimento comparativo definido em `Projeto.md` para **triagem de mensagens informais de atendimento** (9 classes de intenção, sem extração de entidades nesta fase).

Todas as abordagens são avaliadas sobre o mesmo conjunto de teste intocado (**split `test` com 95 amostras** de `RichardSakaguchiMS/brazilian-customer-service-conversations`):
1. **BERTimbau Fine-Tuned** (`modelo_intencao/`)
2. **Gemini Flash-Lite — Zero-shot** (saída JSON estruturada via `google-genai`)
3. **Gemini Flash-Lite — Few-shot** (exemplos estratificados selecionados exclusivamente do split `train`)

Além de Acurácia, Precisão, Revocação e F1-Score, o notebook mede **latência real** e **consumo real de tokens** para projetar custos mensais em escala (1 mil a 1 milhão de mensagens/mês).

## 1. Setup de Ambiente & Configuração (Local e Google Colab)

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    # Ajuste caso utilize um repositório Git remoto ou pasta no Google Drive
    REPO_URL = ""  # Ex.: "https://github.com/usuario/repo.git"
    CANDIDATE_DIRS = [
        Path.cwd(),
        Path("/content/Triagem de chat Ecomerce para LLMs"),
        Path("/content/drive/MyDrive/Triagem de chat para pequenos negocios"),
    ]

    PROJECT_ROOT = next((p for p in CANDIDATE_DIRS if (p / "src").exists()), None)
    if PROJECT_ROOT is None and REPO_URL:
        target = Path("/content/Triagem de chat Ecomerce para LLMs")
        os.system(f"git clone {REPO_URL} {target}")
        PROJECT_ROOT = target

    if PROJECT_ROOT is None:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_ROOT = next(
            (p for p in CANDIDATE_DIRS if (p / "src").exists()),
            Path.cwd(),
        )

    os.chdir(PROJECT_ROOT)
    if (PROJECT_ROOT / "requirements.txt").exists():
        os.system("pip install -q -r requirements.txt")
else:
    cwd = Path.cwd()
    PROJECT_ROOT = cwd if (cwd / "src").exists() else cwd.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Ambiente: {'Colab' if IN_COLAB else 'Local'} | GPU ativa: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    print(f"Ambiente: {'Colab' if IN_COLAB else 'Local'} | Executando em CPU")

try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
except ImportError:
    pass

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY and IN_COLAB:
    try:
        from google.colab import userdata
        GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    except Exception:
        GEMINI_API_KEY = None

Mounted at /content/drive
Ambiente: Colab | GPU ativa: Tesla T4 (14.6 GB VRAM)


In [2]:
# Parâmetros configuráveis do experimento
# Alternativas comuns na família Flash-Lite: "gemini-2.5-flash-lite", "gemini-3.1-flash-lite-preview", "gemini-3.5-flash-lite"
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite-preview")
K_PER_CLASS = 1               # 1 exemplo/classe = 9 exemplos balanceados no prompt few-shot
SEED = 42
INPUT_PRICE_PER_1M = 0.10     # US$ por 1M de tokens de entrada (ajustável)
OUTPUT_PRICE_PER_1M = 0.40    # US$ por 1M de tokens de saída (ajustável)
REQUEST_DELAY_S = 0.2         # Pausa entre requisições para evitar rate limit
USE_MOCK_LLM = False          # Altere para True para validar todo o notebook offline sem API

OUTPUT_DIR = PROJECT_ROOT / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"GEMINI_MODEL: {GEMINI_MODEL} | USE_MOCK_LLM={USE_MOCK_LLM}")
print(f"GEMINI_API_KEY carregada: {'Sim' if GEMINI_API_KEY else 'Não'}")

PROJECT_ROOT: /content
GEMINI_MODEL: gemini-3.1-flash-lite-preview | USE_MOCK_LLM=False
GEMINI_API_KEY carregada: Não


## 2. Verificação Opcional dos IDs de Modelo Disponíveis na Gemini API

Execute esta célula para listar os modelos Flash/Lite habilitados na sua conta e confirmar o valor exato de `GEMINI_MODEL` antes de avaliar as 95 amostras.

In [3]:
if not USE_MOCK_LLM and GEMINI_API_KEY:
    from google import genai

    _client = genai.Client(api_key=GEMINI_API_KEY)
    available_flash_lite = [
        m.name
        for m in _client.models.list()
        if "flash" in m.name.lower() or "lite" in m.name.lower()
    ]
    print("Modelos Flash/Lite disponíveis na conta:")
    for name in sorted(available_flash_lite):
        print(" -", name)
else:
    print("Checagem de modelos pulada (USE_MOCK_LLM=True ou GEMINI_API_KEY ausente).")

Checagem de modelos pulada (USE_MOCK_LLM=True ou GEMINI_API_KEY ausente).


## 3. Carregamento do Dataset & Seleção Estratificada Few-Shot

In [4]:
import pandas as pd
from src.data_loader import INTENT_LABELS, load_intent_splits, select_few_shot_examples

splits = load_intent_splits()
X_train, y_train = splits["train"]
X_val, y_val = splits["validation"]
X_test, y_test = splits["test"]

print(f"Tamanho dos splits -> train: {len(X_train)} | validation: {len(X_val)} | test: {len(X_test)}")
print("\nDistribuição das 9 classes no split de teste (95 amostras):")
display(pd.Series(y_test).value_counts().reindex(INTENT_LABELS, fill_value=0).to_frame("amostras"))

few_shot_examples = select_few_shot_examples(
    train_texts=X_train,
    train_labels=y_train,
    k_per_class=K_PER_CLASS,
    seed=SEED,
)
print(f"\nExemplos few-shot selecionados exclusivamente do train: {len(few_shot_examples)} ({K_PER_CLASS}/classe)")
display(pd.DataFrame(few_shot_examples))

ModuleNotFoundError: No module named 'src'

## 4. Avaliação do BERTimbau Fine-Tuned (`modelo_intencao/`)

In [ ]:
from src.bertimbau_eval import evaluate_bertimbau, load_bertimbau_intent_model
from src.metrics_and_cost import compute_classification_metrics, summarize_latency_and_tokens

bert_model, bert_tokenizer, bert_encoder, bert_device = load_bertimbau_intent_model(
    PROJECT_ROOT / "modelo_intencao"
)
print(f"BERTimbau carregado no dispositivo: {bert_device}")

df_bert = evaluate_bertimbau(
    texts=X_test,
    model=bert_model,
    tokenizer=bert_tokenizer,
    encoder=bert_encoder,
    device=bert_device,
    max_length=128,
    warmup_runs=3,
)

bert_metrics = compute_classification_metrics(y_test, df_bert["pred_label"].tolist())
bert_stats = summarize_latency_and_tokens(df_bert)

print(f"Acurácia: {bert_metrics['accuracy']:.4f} | F1-Macro: {bert_metrics['f1_macro']:.4f}")
print(
    f"Latência -> Média: {bert_stats['latency_mean_ms']:.2f} ms | "
    f"P50: {bert_stats['latency_p50_ms']:.2f} ms | P95: {bert_stats['latency_p95_ms']:.2f} ms"
)
print("\nRelatório de Classificação (BERTimbau):")
print(bert_metrics["report_text"])

## 5. Avaliação do Gemini Flash-Lite — Zero-Shot

In [ ]:
from src.llm_eval import evaluate_gemini

zs_cache_name = "cache_zero_shot_mock.jsonl" if USE_MOCK_LLM else "cache_zero_shot.jsonl"
df_zs = evaluate_gemini(
    texts=X_test,
    mode="zero_shot",
    model_name=GEMINI_MODEL,
    api_key=GEMINI_API_KEY,
    cache_path=OUTPUT_DIR / zs_cache_name,
    request_delay_s=REQUEST_DELAY_S,
    mock=USE_MOCK_LLM,
)

zs_errors = df_zs[df_zs["status"] != "ok"]
if not zs_errors.empty:
    print(f"Atenção: {len(zs_errors)} amostras falharam no Zero-shot:")
    display(zs_errors[["idx", "error"]].head())

zs_metrics = compute_classification_metrics(y_test, df_zs["pred_label"].tolist())
zs_stats = summarize_latency_and_tokens(df_zs)

print(f"Acurácia: {zs_metrics['accuracy']:.4f} | F1-Macro: {zs_metrics['f1_macro']:.4f}")
print(
    f"Latência -> Média: {zs_stats['latency_mean_ms']:.2f} ms | P95: {zs_stats['latency_p95_ms']:.2f} ms"
)
print(
    f"Tokens médios/msg -> Input: {zs_stats['avg_prompt_tokens']:.1f} | "
    f"Output: {zs_stats['avg_output_tokens']:.1f} | Total: {zs_stats['avg_total_tokens']:.1f}"
)
print("\nRelatório de Classificação (Gemini Zero-shot):")
print(zs_metrics["report_text"])

## 6. Avaliação do Gemini Flash-Lite — Few-Shot

In [ ]:
fs_cache_name = "cache_few_shot_mock.jsonl" if USE_MOCK_LLM else "cache_few_shot.jsonl"
df_fs = evaluate_gemini(
    texts=X_test,
    mode="few_shot",
    model_name=GEMINI_MODEL,
    few_shot_examples=few_shot_examples,
    api_key=GEMINI_API_KEY,
    cache_path=OUTPUT_DIR / fs_cache_name,
    request_delay_s=REQUEST_DELAY_S,
    mock=USE_MOCK_LLM,
)

fs_errors = df_fs[df_fs["status"] != "ok"]
if not fs_errors.empty:
    print(f"Atenção: {len(fs_errors)} amostras falharam no Few-shot:")
    display(fs_errors[["idx", "error"]].head())

fs_metrics = compute_classification_metrics(y_test, df_fs["pred_label"].tolist())
fs_stats = summarize_latency_and_tokens(df_fs)

print(f"Acurácia: {fs_metrics['accuracy']:.4f} | F1-Macro: {fs_metrics['f1_macro']:.4f}")
print(
    f"Latência -> Média: {fs_stats['latency_mean_ms']:.2f} ms | P95: {fs_stats['latency_p95_ms']:.2f} ms"
)
print(
    f"Tokens médios/msg -> Input: {fs_stats['avg_prompt_tokens']:.1f} | "
    f"Output: {fs_stats['avg_output_tokens']:.1f} | Total: {fs_stats['avg_total_tokens']:.1f}"
)
print("\nRelatório de Classificação (Gemini Few-shot):")
print(fs_metrics["report_text"])

## 7. Comparação Consolidada, Matrizes de Confusão & Análise de Erros

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from src.metrics_and_cost import build_comparison_table

sns.set_theme(style="whitegrid")

zs_label = f"Gemini {GEMINI_MODEL} (Zero-shot)"
fs_label = f"Gemini {GEMINI_MODEL} (Few-shot)"
bert_label = "BERTimbau (Fine-tuned)"

model_results = {
    bert_label: {"metrics": bert_metrics, "stats": bert_stats},
    zs_label: {"metrics": zs_metrics, "stats": zs_stats},
    fs_label: {"metrics": fs_metrics, "stats": fs_stats},
}

df_comparison = build_comparison_table(model_results)
display(df_comparison)

# Gráficos: Métricas Globais + F1 por Classe
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), gridspec_kw={"width_ratios": [1, 1.35]})

global_melted = df_comparison.melt(
    id_vars="Model",
    value_vars=["Accuracy", "F1 (Macro)", "F1 (Weighted)"],
    var_name="Métrica",
    value_name="Score",
)
sns.barplot(data=global_melted, x="Métrica", y="Score", hue="Model", ax=axes[0])
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Desempenho Global no Split de Teste (n=95)")
axes[0].legend(loc="lower right", fontsize=8)

per_class_f1 = pd.DataFrame(
    {
        bert_label: [bert_metrics["report_dict"][c]["f1-score"] for c in INTENT_LABELS],
        zs_label: [zs_metrics["report_dict"][c]["f1-score"] for c in INTENT_LABELS],
        fs_label: [fs_metrics["report_dict"][c]["f1-score"] for c in INTENT_LABELS],
    },
    index=list(INTENT_LABELS),
)
sns.heatmap(per_class_f1, annot=True, fmt=".2f", cmap="YlGnBu", vmin=0, vmax=1, ax=axes[1])
axes[1].set_title("F1-Score por Classe de Intenção")
axes[1].set_ylabel("Intenção")

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "metrics_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
cms = [
    (bert_label, bert_metrics["confusion_matrix"], "Blues"),
    (zs_label, zs_metrics["confusion_matrix"], "Greens"),
    (fs_label, fs_metrics["confusion_matrix"], "Oranges"),
]

for ax, (title, cm, cmap) in zip(axes, cms):
    sns.heatmap(
        np.array(cm),
        annot=True,
        fmt="d",
        cmap=cmap,
        xticklabels=INTENT_LABELS,
        yticklabels=INTENT_LABELS,
        cbar=False,
        ax=ax,
    )
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Predito")
    ax.set_ylabel("Real")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

# Análise qualitativa de erros e discordâncias
df_errors = pd.DataFrame(
    {
        "idx": range(len(X_test)),
        "mensagem": X_test,
        "real": y_test,
        "bertimbau": df_bert["pred_label"].tolist(),
        "gemini_zero_shot": df_zs["pred_label"].tolist(),
        "gemini_few_shot": df_fs["pred_label"].tolist(),
    }
)
disagreements = df_errors[
    (df_errors["bertimbau"] != df_errors["real"])
    | (df_errors["gemini_zero_shot"] != df_errors["real"])
    | (df_errors["gemini_few_shot"] != df_errors["real"])
]
print(f"Total de amostras com ao menos um erro/discordância: {len(disagreements)} de {len(X_test)}")
display(disagreements.head(20))

## 8. Projeção de Custos e Viabilidade Econômica em Escala

In [ ]:
from src.metrics_and_cost import project_monthly_costs

VOLUMES = (1_000, 10_000, 50_000, 100_000, 500_000, 1_000_000)

df_cost_zs, breakeven_zs = project_monthly_costs(
    avg_prompt_tokens=zs_stats["avg_prompt_tokens"],
    avg_output_tokens=zs_stats["avg_output_tokens"],
    input_price_per_1m=INPUT_PRICE_PER_1M,
    output_price_per_1m=OUTPUT_PRICE_PER_1M,
    volumes=VOLUMES,
)
df_cost_fs, breakeven_fs = project_monthly_costs(
    avg_prompt_tokens=fs_stats["avg_prompt_tokens"],
    avg_output_tokens=fs_stats["avg_output_tokens"],
    input_price_per_1m=INPUT_PRICE_PER_1M,
    output_price_per_1m=OUTPUT_PRICE_PER_1M,
    volumes=VOLUMES,
)

print("=== Projeção Mensal: Gemini Zero-Shot ===")
display(df_cost_zs)
print("Break-even (mensagens/mês) Zero-shot:", breakeven_zs)

print("\n=== Projeção Mensal: Gemini Few-Shot ===")
display(df_cost_fs)
print("Break-even (mensagens/mês) Few-shot:", breakeven_fs)

# Gráfico de escala de custos (escala logarítmica)
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(df_cost_zs["volume_mensal"], df_cost_zs["api_custo_mensal_usd"], marker="o", label=zs_label)
ax.plot(df_cost_fs["volume_mensal"], df_cost_fs["api_custo_mensal_usd"], marker="s", label=fs_label)
ax.axhline(10.0, linestyle="--", color="tab:green", label="BERTimbau VPS CPU (US$ 10/mês)")
ax.axhline(40.0, linestyle=":", color="tab:purple", label="BERTimbau Instância Dedicada (US$ 40/mês)")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Volume Mensal de Mensagens (escala log)")
ax.set_ylabel("Custo Mensal Estimado em US$ (escala log)")
ax.set_title("Projeção de Custo Mensal: BERTimbau Hospedado vs. Gemini Flash-Lite API")
ax.legend()
ax.grid(True, which="both", linestyle="--", alpha=0.4)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "cost_projection.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Exportação dos Artefatos Finais

In [ ]:
import json

df_comparison.to_csv(OUTPUT_DIR / "comparison_table.csv", index=False)
df_bert.to_csv(OUTPUT_DIR / "predictions_bertimbau.csv", index=False)
df_zs.to_csv(OUTPUT_DIR / "predictions_gemini_zero_shot.csv", index=False)
df_fs.to_csv(OUTPUT_DIR / "predictions_gemini_few_shot.csv", index=False)
df_cost_zs.to_csv(OUTPUT_DIR / "cost_projection_zero_shot.csv", index=False)
df_cost_fs.to_csv(OUTPUT_DIR / "cost_projection_few_shot.csv", index=False)
disagreements.to_csv(OUTPUT_DIR / "error_analysis.csv", index=False)

summary_payload = {
    "gemini_model": GEMINI_MODEL,
    "mock_mode": USE_MOCK_LLM,
    "test_size": len(X_test),
    "k_per_class_few_shot": K_PER_CLASS,
    "pricing_usd_per_1m_tokens": {
        "input": INPUT_PRICE_PER_1M,
        "output": OUTPUT_PRICE_PER_1M,
    },
    "comparison_table": df_comparison.to_dict(orient="records"),
    "breakeven_messages_per_month": {
        "zero_shot": breakeven_zs,
        "few_shot": breakeven_fs,
    },
}

summary_path = OUTPUT_DIR / "comparison_summary.json"
summary_path.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Artefatos salvos em: {OUTPUT_DIR}")